In [ ]:
pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv(r'C:\Users\Dell\Downloads\creditcard.csv')


# Summary statistics & missing values check
print("Dataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum().sum())
print("\nClass Distribution:")
print(df['Class'].value_counts())
print("\nClass Proportions (%):")
print(df['Class'].value_counts(normalize=True) * 100)

# Quick look at data
print(df.head())

In [ ]:
from sklearn.preprocessing import RobustScaler

# RobustScaler is less sensitive to extreme outliers in transaction amounts
scaler = RobustScaler()

df['scaled_amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_time'] = scaler.fit_transform(df[['Time']])

# Drop original unscaled columns
df_clean = df.drop(columns=['Amount', 'Time'])

# Plot Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=df_clean, palette=['#1f77b4', '#d62728'])
plt.title('Transaction Class Distribution (0: Legitimate, 1: Fraud)')
plt.yscale('log') # Log scale helps visualize the small minority class
plt.ylabel('Count (Log Scale)')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Define X and y
X = df_clean.drop(columns=['Class'])
y = df_clean['Class']

# Stratified Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Original Train shape: {X_train.shape}, Fraud cases: {sum(y_train == 1)}")

# Apply SMOTE only on Train data to prevent data leakage
smote = SMOTE(sampling_strategy=0.1, random_state=42) # Resample fraud to 10% of majority class
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Resampled Train shape: {X_train_res.shape}, Fraud cases: {sum(y_train_res == 1)}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc

# Initialize Models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
}

# Evaluate Models
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_res, y_train_res)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate Precision-Recall AUC
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(recall, precision)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    results[name] = model
    
    print(f"\n==================== {name} Results ====================")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC Score: {roc_auc:.4f}")
    print(f"PR-AUC Score: {pr_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, model in results.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    
    # PR Curve
    axes[0].plot(recall, precision, label=f'{name}')
    
axes[0].set_xlabel('Recall (Sensitivity)')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend()
axes[0].grid(True)

# Confusion Matrix for XGBoost
y_pred_xgb = results['XGBoost'].predict(X_test)
cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Legitimate', 'Fraud'], yticklabels=['Legitimate', 'Fraud'])
axes[1].set_title('XGBoost Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
importances = results['XGBoost'].feature_importances_
features = X.columns

feat_imp = pd.DataFrame({'Feature': features, 'Importance': importances})
feat_imp = feat_imp.sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x='Importance', y='Feature', data=feat_imp, palette='viridis')
plt.title('Top 10 Important Features for Fraud Detection (XGBoost)')
plt.show()

In [ ]:
# False Positive and False Negative Analysis

from sklearn.metrics import confusion_matrix

# Get the XGBoost model from the results dictionary
xgb_model = results['XGBoost']

# Make predictions
y_pred = xgb_model.predict(X_test)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

TN, FP, FN, TP = cm.ravel()

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

In [ ]:
# False Positive Rate and False Negative Rate

false_positive_rate = FP / (FP + TN)
false_negative_rate = FN / (FN + TP)

print("False Positive Rate:", round(false_positive_rate * 100, 2), "%")
print("False Negative Rate:", round(false_negative_rate * 100, 2), "%")

In [ ]:
# Threshold Analysis for XGBoost

from sklearn.metrics import precision_score, recall_score, f1_score

# Get probability of fraud
y_prob = xgb_model.predict_proba(X_test)[:, 1]

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

threshold_results = []

for threshold in thresholds:
    y_threshold = (y_prob >= threshold).astype(int)

    threshold_results.append({
        'Threshold': threshold,
        'Precision': precision_score(y_test, y_threshold, zero_division=0),
        'Recall': recall_score(y_test, y_threshold, zero_division=0),
        'F1 Score': f1_score(y_test, y_threshold, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    threshold_df['Threshold'],
    threshold_df['Precision'],
    marker='o',
    label='Precision'
)

plt.plot(
    threshold_df['Threshold'],
    threshold_df['Recall'],
    marker='o',
    label='Recall'
)

plt.plot(
    threshold_df['Threshold'],
    threshold_df['F1 Score'],
    marker='o',
    label='F1 Score'
)

plt.xlabel('Classification Threshold')
plt.ylabel('Score')
plt.title('Precision, Recall and F1 Score at Different Thresholds')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Random Forest Model Tuning

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_params,
    n_iter=10,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train_res, y_train_res)

print("Best Random Forest Parameters:")
print(rf_search.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(round(rf_search.best_score_, 4))

In [ ]:
# XGBoost Model Tuning

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=xgb_params,
    n_iter=10,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train_res, y_train_res)

print("Best XGBoost Parameters:")
print(xgb_search.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(round(xgb_search.best_score_, 4))

In [ ]:
# Final Comparison of Tuned Models

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

tuned_models = {
    'Tuned Random Forest': rf_search.best_estimator_,
    'Tuned XGBoost': xgb_search.best_estimator_
}

final_results = []

for name, model in tuned_models.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    final_results.append({
        'Model': name,
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

final_results_df = pd.DataFrame(final_results)

final_results_df

In [ ]:
# Confusion Matrix for Tuned Models

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, model) in zip(axes, tuned_models.items()):

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=ax,
        xticklabels=['Legitimate', 'Fraud'],
        yticklabels=['Legitimate', 'Fraud']
    )

    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
## Final Observations

- The dataset contains a highly imbalanced number of legitimate and fraudulent transactions.
- SMOTE was applied to the training data to address the class imbalance problem.
- Random Forest and XGBoost classification models were used for fraud detection.
- Model performance was evaluated using Precision, Recall, F1 Score and ROC-AUC.
- Confusion matrices were used to identify True Positives, True Negatives, False Positives and False Negatives.
- False Positive analysis helps identify legitimate transactions that may be incorrectly flagged as fraud.
- False Negative analysis helps identify fraudulent transactions that the model fails to detect.
- Feature importance analysis was used to identify the variables that contributed most to the fraud predictions.
- Model tuning was performed using RandomizedSearchCV to find better model parameters.
- Different

In [ ]:
## Recommendations

1. **Use an appropriate classification threshold:**  
   The threshold can be adjusted according to the required balance between detecting fraud and avoiding unnecessary fraud alerts.

2. **Monitor false negatives:**  
   Fraudulent transactions that are incorrectly classified as legitimate should be monitored because they may result in financial losses.

3. **Reduce false positives:**  
   Excessive false alerts can inconvenience genuine customers, so the model should be monitored to maintain a suitable precision level.

4. **Regularly retrain the model:**  
   Fraud patterns can change over time. The model should therefore be updated using recent transaction data.

5. **Real-time fraud monitoring:**  
   A practical fraud detection system can assign a risk score to transactions in real time and flag suspicious transactions for further verification.

6. **Human verification for suspicious transactions:**  
   High-risk transactions can be sent for additional verification rather than automatically rejecting every suspicious transaction.

7. **Continuous performance monitoring:**  
   Precision, recall, F1-score and false positive/negative rates should be monitored regularly after deployment.

In [ ]:
## Conclusion

The project developed a machine learning based approach for detecting fraudulent credit card transactions. The data was explored and preprocessed, and the class imbalance problem was addressed using SMOTE. Random Forest and XGBoost models were trained and evaluated using Precision, Recall, F1 Score and ROC-AUC.

Confusion matrix analysis was used to study false positives and false negatives, while feature importance helped identify the variables contributing to fraud predictions. Model tuning and threshold analysis were also performed to improve the understanding of model performance.

The results demonstrate how machine learning can be used as a supporting tool for identifying potentially fraudulent transactions and improving transaction risk monitoring.